# 特徵值與特徵向量 (Eigenvalues and Eigenvectors)

本筆記本使用 **NumPy** 介紹線性代數中最重要的概念之一——**特徵值與特徵向量**：

1. **定義** — $A\mathbf{v}=\lambda\mathbf{v}$ 是什麼意思？
2. **幾何意義** — 矩陣如何「拉伸／翻轉」特定方向
3. **特徵多項式** — $\det(A-\lambda I)=0$
4. **逐步手算** — 用數值方法跟一次完整流程
5. **`eig` / `eigvals` / `eigh`** — NumPy 快捷方法
6. **特徵空間與重數** — 代數重數 vs 幾何重數
7. **對角化** — $A=PDP^{-1}$
8. **對稱矩陣** — 實對稱 ⇒ 正交對角化
9. **應用** — 用對角化算 $A^n$
10. **綜合練習**

> NumPy 做的是**數值**特徵分解（浮點近似），不是符號精確解。需要精確分數／符號時請看同目錄的 `eigen_sympy.ipynb`。

In [1]:
import numpy as np
from IPython.display import display, Math


def _fmt_number(x):
    """Format a number for LaTeX display."""
    x = np.asarray(x).item()
    if isinstance(x, complex) or np.iscomplexobj(x):
        re, im = float(np.real(x)), float(np.imag(x))
        if abs(im) < 1e-10:
            x = re
        else:
            re_s = _fmt_number(re)
            im_s = _fmt_number(abs(im))
            sign = "+" if im >= 0 else "-"
            return f"{re_s} {sign} {im_s} i"
    x = float(np.real(x))
    if abs(x - round(x)) < 1e-10:
        return str(int(round(x)))
    return f"{x:.6g}"


def _array_to_latex(arr):
    """Convert a numpy array / scalar to a LaTeX string."""
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return _fmt_number(arr)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    rows = []
    for row in arr:
        rows.append(" & ".join(_fmt_number(v) for v in row))
    body = r"\\".join(rows)
    return r"\begin{bmatrix}" + body + r"\end{bmatrix}"


def display_large(expr):
    """Display a numpy array / scalar in LaTeX with large font size."""
    display(Math(r"\large{%s}" % _array_to_latex(expr)))


def display_huge(expr):
    """Display a numpy array / scalar in LaTeX with huge font size."""
    display(Math(r"\huge{%s}" % _array_to_latex(expr)))

## 1. 什麼是特徵值與特徵向量？

給定方陣 $A$（$n\times n$），若存在**非零**向量 $\mathbf{v}$ 與純量 $\lambda$，使得

$$
A\mathbf{v} = \lambda\mathbf{v}
$$

則稱：

| 名稱 | 英文 | 符號 |
|------|------|------|
| **特徵值** | eigenvalue | $\lambda$ |
| **特徵向量** | eigenvector | $\mathbf{v}$（$\mathbf{v}\neq\mathbf{0}$） |

直觀意義：對特徵向量做線性變換 $A$，結果只是**沿同一方向伸縮**（乘上 $\lambda$），**方向不變**（若 $\lambda<0$ 則反向）。

等價改寫：

$$
(A-\lambda I)\mathbf{v} = \mathbf{0}
$$

也就是：非零 $\mathbf{v}$ 落在 $A-\lambda I$ 的**零空間**裡。因此 $\lambda$ 是特徵值 $\Leftrightarrow$ $A-\lambda I$ **奇異** $\Leftrightarrow$

$$
\det(A-\lambda I) = 0
$$

In [2]:
# Quick check: Av = λv for a known pair
A = np.array([[2.0, 1.0], [1.0, 2.0]])
v = np.array([[1.0], [1.0]])
lam = 3.0

print("A =")
display_huge(A)
print("v =")
display_huge(v)
print("A v =")
display_huge(A @ v)
print("λ v =")
display_huge(lam * v)
print("Av ≈ λv?", np.allclose(A @ v, lam * v))

A =


<IPython.core.display.Math object>

v =


<IPython.core.display.Math object>

A v =


<IPython.core.display.Math object>

λ v =


<IPython.core.display.Math object>

Av ≈ λv? True


## 2. 幾何意義

對同一個矩陣 $A$，不同方向的向量被「拉」的程度不同：

- 特徵方向：$A\mathbf{v}$ 與 $\mathbf{v}$ **平行**
- 一般方向：$A\mathbf{w}$ 會**旋轉＋伸縮**，不再平行於 $\mathbf{w}$

$\lambda$ 告訴你伸縮倍率：

| $\lambda$ | 效果 |
|-----------|------|
| $\lambda>1$ | 沿該方向拉長 |
| $0<\lambda<1$ | 沿該方向縮短 |
| $\lambda=1$ | 該方向不動 |
| $\lambda=0$ | 壓成零向量（落在零空間） |
| $\lambda<0$ | 反向再伸縮 |

In [3]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])

# Eigen-direction: stays parallel
v_eigen = np.array([[1.0], [1.0]])
# Generic direction: gets rotated
w = np.array([[1.0], [0.0]])

Av = A @ v_eigen
Aw = A @ w


def is_parallel(u, v, tol=1e-10):
    """Check if 2D column vectors u, v are parallel (cross product ~ 0)."""
    return abs(u[0, 0] * v[1, 0] - u[1, 0] * v[0, 0]) < tol


print("Eigen direction v = [1, 1]^T")
print("A v =")
display_huge(Av)
print("Parallel to v?", is_parallel(Av, v_eigen))

print("\nGeneric direction w = [1, 0]^T")
print("A w =")
display_huge(Aw)
print("Parallel to w?", is_parallel(Aw, w))

Eigen direction v = [1, 1]^T
A v =


<IPython.core.display.Math object>

Parallel to v? True

Generic direction w = [1, 0]^T
A w =


<IPython.core.display.Math object>

Parallel to w? False


## 3. 特徵多項式

定義**特徵多項式（characteristic polynomial）**為

$$
p_A(\lambda) = \det(\lambda I - A)
$$

（有些書寫成 $\det(A-\lambda I)$，只差一個符號 $(-1)^n$。）

特徵值就是 $p_A(\lambda)=0$ 的根。對 $n\times n$ 矩陣，$p_A$ 是 $n$ 次多項式，在複數體上恰有 $n$ 個根（計重數）。

NumPy：

- `np.poly(A)` — 特徵多項式係數（對應 $\det(\lambda I-A)$，最高次項在前）
- `np.roots(coeffs)` — 多項式的根
- `np.linalg.eigvals(A)` — 直接求特徵值（數值上更穩定，實務首選）

In [4]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])

# Characteristic polynomial of det(λI - A): λ^2 - 4λ + 3
coeffs = np.poly(A)
print("A =")
display_huge(A)
print("Characteristic polynomial coeffs [λ^n, ..., const]:")
display_large(coeffs)
print("→ λ² - 4λ + 3 = 0")

print("\nRoots via np.roots:")
display_large(np.roots(coeffs))

print("Eigenvalues via np.linalg.eigvals:")
display_large(np.linalg.eigvals(A))

A =


<IPython.core.display.Math object>

Characteristic polynomial coeffs [λ^n, ..., const]:


<IPython.core.display.Math object>

→ λ² - 4λ + 3 = 0

Roots via np.roots:


<IPython.core.display.Math object>

Eigenvalues via np.linalg.eigvals:


<IPython.core.display.Math object>

## 4. 逐步手算：從定義到特徵向量

標準流程（以 $2\times 2$ 為例）：

1. 算 $A-\lambda I$
2. 令 $\det(A-\lambda I)=0$，解出 $\lambda$
3. 對每個 $\lambda$，解 $(A-\lambda I)\mathbf{v}=\mathbf{0}$（找零空間）
4. 得到特徵空間 $E_\lambda = N(A-\lambda I)$

在 NumPy 中：用 `eigvals` 得 $\lambda$，再用 SVD 找 $A-\lambda I$ 的數值零空間。

In [5]:
A = np.array([[4.0, 2.0], [1.0, 3.0]])

print("Step 0: A =")
display_huge(A)

# Step 1–2: characteristic equation via np.poly / eigvals
coeffs = np.poly(A)
print("Step 1–2: characteristic polynomial coeffs [λ^n, ..., const]:")
display_large(coeffs)

eigs = np.linalg.eigvals(A)
# Sort for stable display (real parts first)
eigs = np.sort_complex(eigs)
print("Eigenvalues:")
display_large(eigs)

Step 0: A =


<IPython.core.display.Math object>

Step 1–2: characteristic polynomial coeffs [λ^n, ..., const]:


<IPython.core.display.Math object>

Eigenvalues:


<IPython.core.display.Math object>

In [6]:
def nullspace(M, tol=1e-10):
    """Numerical nullspace basis via SVD (columns of V^T corresponding to tiny singular values)."""
    U, s, Vt = np.linalg.svd(M)
    rank = (s > tol).sum()
    return Vt[rank:].T


# Step 3–4: nullspace for each eigenvalue
for eig in np.real_if_close(eigs):
    eig = float(np.real(eig))
    print(f"\n--- λ = {eig:g} ---")
    N = A - eig * np.eye(2)
    print("A - λI =")
    display_huge(N)
    print("det(A - λI) ≈", np.linalg.det(N))
    basis = nullspace(N)
    print("Nullspace basis (eigenvector columns):")
    display_huge(basis)
    for j in range(basis.shape[1]):
        v = basis[:, [j]]
        print("Check A v ≈ λ v:")
        display_large(np.hstack([A @ v, eig * v]))
        print("allclose?", np.allclose(A @ v, eig * v))


--- λ = 2 ---
A - λI =


<IPython.core.display.Math object>

det(A - λI) ≈ 0.0
Nullspace basis (eigenvector columns):


<IPython.core.display.Math object>

Check A v ≈ λ v:


...

det(A - λI) ≈ 0.0
Nullspace basis (eigenvector columns):


<IPython.core.display.Math object>

Check A v ≈ λ v:


<IPython.core.display.Math object>

allclose? True


## 5. NumPy 快捷：`eig`、`eigvals`、`eigh`

| 函式 | 用途 |
|------|------|
| `np.linalg.eigvals(A)` | 只回傳特徵值 |
| `np.linalg.eig(A)` | 回傳 `(eigenvalues, eigenvectors)`，向量為**欄** |
| `np.linalg.eigh(A)` | 專給 **Hermitian／實對稱**，特徵值保證實數且由小到大 |

`eig` 回傳的特徵向量通常已**正規化**（長度約為 1），符號／縮放與手算基底可能不同，但張成同一特徵空間。

In [7]:
A = np.array([[4.0, 2.0], [1.0, 3.0]])

print("eigvals():")
display_large(np.linalg.eigvals(A))

evals, evecs = np.linalg.eig(A)
print("\neig() → eigenvalues:")
display_large(evals)
print("eigenvectors (columns):")
display_huge(evecs)

for i, lam in enumerate(evals):
    v = evecs[:, [i]]
    print(f"λ = {lam:g}, ||v|| = {np.linalg.norm(v):.6g}")
    print("A v ≈ λ v?", np.allclose(A @ v, lam * v))

eigvals():


<IPython.core.display.Math object>


eig() → eigenvalues:


<IPython.core.display.Math object>

eigenvectors (columns):


<IPython.core.display.Math object>

λ = 5+0j, ||v|| = 1
A v ≈ λ v? True
λ = 2+0j, ||v|| = 1
A v ≈ λ v? True


## 6. 特徵空間與重數

對特徵值 $\lambda$：

| 概念 | 定義 |
|------|------|
| **特徵空間** $E_\lambda$ | $N(A-\lambda I)=\{\mathbf{v}:A\mathbf{v}=\lambda\mathbf{v}\}$（含 $\mathbf{0}$） |
| **代數重數** | 特徵多項式中 $(\lambda-\lambda_0)$ 的次數 |
| **幾何重數** | $\dim E_\lambda$（獨立特徵向量個數） |

永遠有

$$
1 \le \text{幾何重數} \le \text{代數重數}
$$

若某個 $\lambda$ 的幾何重數 **小於** 代數重數，矩陣**不可對角化**（defectiveness）。

數值上：幾何重數 ≈ `nullspace(A - λI)` 的欄數；若 `eig` 的特徵向量矩陣條件數很大，常暗示缺陷（缺特征向量）。

In [8]:
def geometric_multiplicity(A, lam, tol=1e-8):
    """Estimate geometric multiplicity as dim N(A - λI)."""
    N = A - lam * np.eye(A.shape[0])
    return nullspace(N, tol=tol).shape[1]


# Repeated eigenvalue with full geometric multiplicity
A_good = np.array([
    [2.0, 0.0, 0.0],
    [0.0, 2.0, 0.0],
    [0.0, 0.0, 3.0],
])
print("A (diagonalizable):")
display_huge(A_good)
evals = np.linalg.eigvals(A_good)
print("eigvals:")
display_large(evals)

unique, counts = np.unique(np.round(np.real(evals), 10), return_counts=True)
for lam, alg in zip(unique, counts):
    geo = geometric_multiplicity(A_good, lam)
    print(f"λ={lam:g}: alg≈{alg}, geo≈{geo}")

# Defective: Jordan block — only one independent eigenvector for λ=2 (alg=2)
A_bad = np.array([[2.0, 1.0], [0.0, 2.0]])
print("\nA (defective, not diagonalizable):")
display_huge(A_bad)
print("Characteristic polynomial coeffs:")
display_large(np.poly(A_bad))

evals_bad, evecs_bad = np.linalg.eig(A_bad)
print("eigvals:")
display_large(evals_bad)
print("eigvecs (columns) — nearly parallel when defective:")
display_huge(evecs_bad)
print("cond(P) =", np.linalg.cond(evecs_bad))
print("geo multiplicity of λ=2 ≈", geometric_multiplicity(A_bad, 2.0))

A (diagonalizable):


<IPython.core.display.Math object>

eigvals:


<IPython.core.display.Math object>

λ=2: alg≈2, geo≈2
λ=3: alg≈1, geo≈1

A (defective, not diagonalizable):


...

eigvals:


<IPython.core.display.Math object>

eigvecs (columns) — nearly parallel when defective:


<IPython.core.display.Math object>

cond(P) = 4503599627370495.5
geo multiplicity of λ=2 ≈ 1


## 7. 對角化 $A = PDP^{-1}$

若 $A$ 有 $n$ 個**線性獨立**的特徵向量 $\mathbf{v}_1,\ldots,\mathbf{v}_n$（對應 $\lambda_1,\ldots,\lambda_n$），令

$$
P = [\mathbf{v}_1\;\cdots\;\mathbf{v}_n],\qquad
D = \operatorname{diag}(\lambda_1,\ldots,\lambda_n)
$$

則

$$
AP = PD \quad\Rightarrow\quad A = PDP^{-1}
$$

判別：$A$ 可對角化 $\Leftrightarrow$ 每個特徵值的幾何重數 = 代數重數 $\Leftrightarrow$ 存在完整的特徵向量基底。

NumPy：`evals, P = np.linalg.eig(A)`，再令 `D = np.diag(evals)`。

In [9]:
A = np.array([[4.0, 2.0], [1.0, 3.0]])

evals, P = np.linalg.eig(A)
D = np.diag(evals)

print("P (columns = eigenvectors):")
display_huge(P)
print("D (eigenvalues on diagonal):")
display_huge(D)

print("Check P^{-1} A P ≈ D:")
display_huge(np.linalg.inv(P) @ A @ P)

print("Check A ≈ P D P^{-1}:")
A_recon = P @ D @ np.linalg.inv(P)
display_huge(A_recon)
print("Match original A?", np.allclose(A_recon, A))

P (columns = eigenvectors):


<IPython.core.display.Math object>

D (eigenvalues on diagonal):


<IPython.core.display.Math object>

Check P^{-1} A P ≈ D:


<IPython.core.display.Math object>

Check A ≈ P D P^{-1}:


<IPython.core.display.Math object>

Match original A? True


In [10]:
# Defective matrix: eig still returns something, but P is nearly singular
A_bad = np.array([[2.0, 1.0], [0.0, 2.0]])
evals, P = np.linalg.eig(A_bad)
print("defective A =")
display_huge(A_bad)
print("P from eig:")
display_huge(P)
print("cond(P) =", np.linalg.cond(P))
print("P is (numerically) singular / ill-conditioned → not stably diagonalizable")

# Reconstruction may look ok in float noise for this tiny example,
# but large cond(P) signals the algebraic failure of diagonalization.
try:
    A_recon = P @ np.diag(evals) @ np.linalg.inv(P)
    print("||A - P D P^{-1}|| ≈", np.linalg.norm(A - A_recon))
except np.linalg.LinAlgError as e:
    print("inv(P) failed:", e)

defective A =


<IPython.core.display.Math object>

P from eig:


<IPython.core.display.Math object>

cond(P) = 4503599627370495.5
P is (numerically) singular / ill-conditioned → not stably diagonalizable
||A - P D P^{-1}|| ≈ 3.1622776601683795


## 8. 對稱矩陣：正交對角化

**實對稱**矩陣（$A^{\mathsf{T}}=A$）有漂亮性質：

1. 所有特徵值都是**實數**
2. 不同特徵值的特徵向量**彼此正交**
3. 一定可以**正交對角化**：$A=QDQ^{\mathsf{T}}$，其中 $Q^{\mathsf{T}}Q=I$

NumPy 請用 **`np.linalg.eigh`**（比一般 `eig` 更穩定，且特徵值已排序）。

In [11]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
print("A (symmetric):")
display_huge(A)
print("A^T = A?", np.allclose(A.T, A))

evals, Q = np.linalg.eigh(A)
D = np.diag(evals)

print("Eigenvalues (all real, sorted ascending):")
display_large(evals)

print("Q (orthonormal columns):")
display_huge(Q)
print("v1 · v2 =")
display_large(Q[:, 0] @ Q[:, 1])

print("Q^T Q ≈ I?")
display_huge(Q.T @ Q)
print("allclose to I?", np.allclose(Q.T @ Q, np.eye(2)))

print("A ≈ Q D Q^T?")
display_huge(Q @ D @ Q.T)
print("Match?", np.allclose(Q @ D @ Q.T, A))

A (symmetric):


<IPython.core.display.Math object>

A^T = A? True
Eigenvalues (all real, sorted ascending):


<IPython.core.display.Math object>

Q (orthonormal columns):


...

Q^T Q ≈ I?


<IPython.core.display.Math object>

allclose to I? True
A ≈ Q D Q^T?


<IPython.core.display.Math object>

Match? True


## 9. 應用：用對角化算 $A^n$

若 $A=PDP^{-1}$，則

$$
A^n = PD^n P^{-1}
$$

而 $D^n$ 只是把對角元各自取 $n$ 次方——比直接乘矩陣 $n$ 次省事得多。

（符號閉式 $A^n$ 請用 SymPy；這裡用數值驗證 $n=5$。）

In [12]:
A = np.array([[4.0, 2.0], [1.0, 3.0]])
evals, P = np.linalg.eig(A)
D = np.diag(evals)

n = 5
D_n = np.diag(evals ** n)
A_n = P @ D_n @ np.linalg.inv(P)

print("D =")
display_huge(D)
print(f"D^{n} =")
display_huge(D_n)
print(f"A^{n} = P D^{n} P^{{-1}} =")
display_huge(A_n)

print(f"Direct matrix_power(A, {n}) =")
A_direct = np.linalg.matrix_power(A, n)
display_huge(A_direct)
print("Match?", np.allclose(A_n, A_direct))

D =


<IPython.core.display.Math object>

D^5 =


<IPython.core.display.Math object>

A^5 = P D^5 P^{-1} =


<IPython.core.display.Math object>

Direct matrix_power(A, 5) =


<IPython.core.display.Math object>

Match? True


## 10. 綜合練習

考慮

$$
A = \begin{bmatrix} 1 & 2 & 0 \\ 2 & 1 & 0 \\ 0 & 0 & 3 \end{bmatrix}
$$

1. 確認 $A$ 是否對稱
2. 求所有特徵值與對應特徵向量
3. 對每個特徵值，比較代數重數與幾何重數
4. 對角化：$A=PDP^{-1}$（或正交對角化 $QDQ^{\mathsf{T}}$）
5. 用對角化算出 $A^3$，並與直接計算比對

In [13]:
A = np.array([
    [1.0, 2.0, 0.0],
    [2.0, 1.0, 0.0],
    [0.0, 0.0, 3.0],
])

# 1. Symmetric?
print("1. A =")
display_huge(A)
print("Symmetric?", np.allclose(A.T, A))

# 2–3. Eigenpairs and multiplicities (use eigh for symmetric)
evals, Q = np.linalg.eigh(A)
print("\n2–3. Eigenvalues / multiplicities:")
unique, counts = np.unique(np.round(evals, 10), return_counts=True)
for lam, alg in zip(unique, counts):
    geo = geometric_multiplicity(A, lam)
    print(f"λ = {lam:g}, alg ≈ {alg}, geo ≈ {geo}")

print("Eigenvectors (columns of Q):")
display_huge(Q)

# 4. Orthogonal diagonalization
D = np.diag(evals)
print("\n4. Q =")
display_huge(Q)
print("D =")
display_huge(D)
print("Q^T Q ≈ I?", np.allclose(Q.T @ Q, np.eye(3)))
print("A ≈ Q D Q^T?", np.allclose(Q @ D @ Q.T, A))

# 5. A^3 via diagonalization
A3_via = Q @ (D ** 3) @ Q.T
print("\n5. A^3 via Q D^3 Q^T =")
display_huge(A3_via)
print("Direct matrix_power(A, 3) =")
display_huge(np.linalg.matrix_power(A, 3))
print("Match?", np.allclose(A3_via, np.linalg.matrix_power(A, 3)))

1. A =


<IPython.core.display.Math object>

Symmetric? True

2–3. Eigenvalues / multiplicities:
λ = -1, alg ≈ 1, geo ≈ 1
λ = 3, alg ≈ 2, geo ≈ 2
Eigenvectors (columns of Q):


<IPython.core.display.Math object>


4. Q =


...

Q^T Q ≈ I? True
A ≈ Q D Q^T? True

5. A^3 via Q D^3 Q^T =


<IPython.core.display.Math object>

Direct matrix_power(A, 3) =


<IPython.core.display.Math object>

Match? True


## 小結

| 概念 | NumPy / 公式 | 重點 |
|------|--------------|------|
| 定義 | $A\mathbf{v}=\lambda\mathbf{v}$ | $\mathbf{v}\neq\mathbf{0}$，方向不變 |
| 特徵方程 | `np.poly(A)` / `eigvals` | 根 = 特徵值（數值） |
| 特徵向量 | SVD 零空間 / `eig` 的欄 | $E_\lambda=N(A-\lambda I)$ |
| 快捷 | `eig` / `eigvals` / `eigh` | 對稱用 `eigh` |
| 重數 | 代數 ≥ 幾何 ≥ 1 | 幾何 < 代數 ⇒ 不可對角化 |
| 對角化 | $P,D$ from `eig` → $PDP^{-1}$ | 需 $n$ 個獨立特徵向量 |
| 對稱矩陣 | `eigh` → $QDQ^{\mathsf{T}}$ | 特徵向量可正交 |
| 矩陣次方 | $A^n=PD^nP^{-1}$ | 對角元各自乘方 |

**記住：**

- 特徵值／特徵向量描述「矩陣只做伸縮、不旋轉」的特殊方向
- 找 $\lambda$：解 $\det(A-\lambda I)=0$；找 $\mathbf{v}$：解零空間
- 可對角化 $\Leftrightarrow$ 有完整特徵向量基底
- 實對稱矩陣一定可以正交對角化；NumPy 用 `eigh`
- 數值結果是浮點近似；精確符號推導請用 SymPy